In [11]:
%%writefile players.py
import random

FB_MORE = "больше"
FB_LESS = "меньше"
FB_CORRECT = "угадал"

BOT_STRATEGIES = ("dichotomy", "random", "seq")


class Player:
    """Игрок. kind — 'dichotomy', 'random', 'seq' или 'user'."""

    def __init__(self, name, kind, low, high):
        self.name = name
        self.kind = kind
        self.low = low
        self.high = high
        self.last = None

        if kind == 'user':
            self.secret = int(input(f"{self.name}, загадайте число [{low}, {high}]: "))
        else:
            self.secret = random.randint(low, high)

        # начальное состояние стратегии
        self.cur_low = low
        self.cur_high = high
        self.used = set()
        self.current = low

    def make_guess(self):
        if self.kind == 'dichotomy':
            self.last = (self.cur_low + self.cur_high) // 2

        elif self.kind == 'random':
            while True:
                g = random.randint(self.low, self.high)
                if g not in self.used or len(self.used) >= self.high - self.low + 1:
                    self.last = g
                    self.used.add(g)
                    break

        elif self.kind == 'seq':
            self.last = self.current
            self.current = self.current + 1 if self.current < self.high else self.low

        elif self.kind == 'user':
            self.last = int(input(f"{self.name}, ваш ход [{self.low}, {self.high}]: "))

        return self.last

    def feedback(self, fb):
        if self.kind == 'dichotomy':
            if fb == FB_MORE:
                self.cur_low = self.last + 1
            elif fb == FB_LESS:
                self.cur_high = self.last - 1

Overwriting players.py


In [12]:
%%writefile manager.py
from players import Player, FB_MORE, FB_LESS, FB_CORRECT, BOT_STRATEGIES


class Game:
    """Менеджер игры."""

    def __init__(self):
        self.players = []
        self.low = None
        self.high = None
        self.k = None

    def setup(self):
        self.low = int(input("нижняя граница: "))
        self.high = int(input("верхняя граница: "))
        self.k = int(input("раундов (K): "))

    def add_player(self, name, kind):
        p = Player(name, kind, self.low, self.high)
        self.players.append(p)
        return p

    def compare(self, guess, secret):
        if guess == secret:
            return FB_CORRECT
        return FB_MORE if guess < secret else FB_LESS

    def play(self):
        p1, p2 = self.players[0], self.players[1]

        print("=" * 40)
        print(f"интервал [{self.low}, {self.high}], раундов {self.k}")
        for p in self.players:
            if p.kind != 'user':
                print(f"{p.name}: стратегия={p.kind}")
            else:
                print(f"{p.name}")
        print("=" * 40)

        for step in range(1, self.k + 1):
            print(f"\nраунд {step}")

            g = p1.make_guess()
            fb = self.compare(g, p2.secret)
            print(f"{p1.name} -> {g}: {fb}")
            p1.feedback(fb)
            if fb == FB_CORRECT:
                self._finish(p1, "угадал")
                return

            g = p2.make_guess()
            fb = self.compare(g, p1.secret)
            print(f"{p2.name} -> {g}: {fb}")
            p2.feedback(fb)
            if fb == FB_CORRECT:
                self._finish(p2, "угадал")
                return

        d1 = abs(p1.last - p2.secret)
        d2 = abs(p2.last - p1.secret)
        if d1 < d2:
            self._finish(p1, "ближе по расстоянию")
        elif d2 < d1:
            self._finish(p2, "ближе по расстоянию")
        else:
            print("\nничья")
            self._finish(None, "")

    def _finish(self, winner, reason):
        p1, p2 = self.players
        d1 = abs(p1.last - p2.secret)
        d2 = abs(p2.last - p1.secret)

        if winner is not None:
            print(f"\nпобедил {winner.name} ({reason})")

        print(f"расстояние {p1.name}: {d1}")
        print(f"расстояние {p2.name}: {d2}")
        print("загаданные числа:")
        for p in self.players:
            print(f"  {p.name}: {p.secret}")

Overwriting manager.py


In [13]:
%%writefile main.py
from manager import Game
from players import BOT_STRATEGIES


def ask_bot_strategy(label):
    print(f"стратегии бота для {label}: {', '.join(BOT_STRATEGIES)}")
    while True:
        k = input(f"{label}: ").strip()
        if k in BOT_STRATEGIES:
            return k
        print("неверная стратегия, попробуйте снова")


def main():
    game = Game()
    game.setup()

    print("\nрежим: 1 — бот-бот, 2 — бот-человек")
    mode = input("выбор: ")

    if mode == "1":
        k1 = ask_bot_strategy("стратегия игрока 1")
        k2 = ask_bot_strategy("стратегия игрока 2")
        game.add_player("Игрок-1", k1)
        game.add_player("Игрок-2", k2)
    else:
        k1 = ask_bot_strategy("стратегия бота")
        game.add_player("Бот", k1)
        game.add_player("Пользователь", "user")

    game.play()


if __name__ == "__main__":
    main()

Overwriting main.py


In [14]:
!python main.py

нижняя граница: 1
верхняя граница: 100
раундов (K): 5

режим: 1 — бот-бот, 2 — бот-человек
выбор: 1
стратегии бота для стратегия игрока 1: dichotomy, random, seq
стратегия игрока 1: dichotomy
стратегии бота для стратегия игрока 2: dichotomy, random, seq
стратегия игрока 2: dichotomy
интервал [1, 100], раундов 5
Игрок-1: стратегия=dichotomy
Игрок-2: стратегия=dichotomy

раунд 1
Игрок-1 -> 50: больше
Игрок-2 -> 50: больше

раунд 2
Игрок-1 -> 75: меньше
Игрок-2 -> 75: больше

раунд 3
Игрок-1 -> 62: больше
Игрок-2 -> 88: меньше

раунд 4
Игрок-1 -> 68: угадал

победил Игрок-1 (угадал)
расстояние Игрок-1: 0
расстояние Игрок-2: 3
загаданные числа:
  Игрок-1: 85
  Игрок-2: 68
